# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their `@id` as per the [Croissant schema](https://mlcommons.org/croissant/).

### Dataset Source
The dataset is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install -q mlcroissant matplotlib seaborn

## 1. Data Loading

Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set your Croissant schema URL here
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}\n")
print(metadata.description)


## 2. Data Overview

Let's review the available record sets and their associated field and column `@id`s. All references use Croissant schema entity `@id` for accuracy and reproducibility.

**List available record sets and their field `@id`s:**

In [ ]:
# Gather record sets and show their @id and associated fields
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}\n  Name: {rs.name}\n  Description: {rs.description}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - Field @id: {f.id}, Name: {f.name}, Data type: {f.data_type}")
    print('')

## 3. Data Extraction

Extract data from each available record set. All references use the `@id` of the respective record set.

*This code will automatically extract and preview the first few rows from each record set.*

In [ ]:
dfs = {}
for rs in record_sets:
    print(f"Loading {rs.id} ...")
    df = pd.DataFrame(dataset.records(record_set=rs.id))
    dfs[rs.id] = df
    print(f"Columns: {list(df.columns)}\nPreview:")
    display(df.head())


## 4. Exploratory Data Analysis (EDA)

We'll demonstrate EDA using the main protein quantitation record set. Replace the values below with the actual `@id` of the record set and fields you want to analyze as per the printed results above.

**Example: Filtering by molecular weight (`MW`), normalizing, and grouping by a categorical field (e.g., protein description).**

In [ ]:
# Update these variables using @id from cell above if available
# Example IDs (replace according to your dataset):
#   record_set_id = 'protein_quantitation'
#   numeric_field_id = 'MW'
#   group_field_id = 'Description'

# For this example, auto-detect a numeric field and a group field
record_set_id = None
numeric_field_id = None
group_field_id = None

# Try to find the largest DataFrame
main_rs_id = max(dfs, key=lambda k: len(dfs[k]) if len(dfs[k]) > 0 else 0)
df = dfs[main_rs_id]
record_set_id = main_rs_id

# Attempt to pick a numeric field
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
# Attempt to pick a group field (categorical)
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() < 20:
        group_field_id = col
        break

print(f"Using RecordSet @id: {record_set_id}\nNumeric field: {numeric_field_id}\nGroup field: {group_field_id}\n")

if numeric_field_id:
    threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id:
        # Group and show mean of numeric field
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by '{group_field_id}':")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relationship to the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR² Croissant dataset using the `mlcroissant` library. We listed record sets, extracted data by referencing entity `@id`, and applied basic EDA and visualization techniques. For advanced analyses, consult the specific column and record set documentation accessed by `@id` as shown in this notebook.

**Remember:** Always reference Croissant entities using their canonical `@id` to enable consistent, reproducible analyses.
